# Monte Carlo tests

In [1]:
import numpy as np
import pandas as pd
import sys
import plotly.express as px
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.utils import load_config
from src.monte_carlo import run_simulation_grid
from src.metrics import summarize_df, rmse_diff

c:\Users\danil\anaconda3\envs\vcausal\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Running MC simulations

In [2]:
config = load_config("baseline")

df = run_simulation_grid(config)

alpha_y=1, alpha_d=1: 100%|██████████| 50/50 [01:13<00:00,  1.46s/it]


In [3]:
df.head()

,alpha_y,alpha_d,seed,tau_true,ols_tau_hat,ols_se,ols_ci_lower,ols_ci_upper,dml_tau_hat,dml_se,dml_ci_lower,dml_ci_upper,replication
0,0,0,42,2.5,2.568462,0.087267,2.397418,2.739506,2.555995,0.101172,2.357699,2.754291,0
1,0,0,43,2.5,2.560770,0.091364,2.381697,2.739843,2.668774,0.111203,2.450815,2.886733,1
2,0,0,44,2.5,2.403229,0.093729,2.219521,2.586937,2.485289,0.106290,2.276960,2.693618,2
3,0,0,45,2.5,2.487000,0.091947,2.306784,2.667217,2.551015,0.108989,2.337396,2.764633,3
4,0,0,46,2.5,2.354004,0.090233,2.177148,2.530859,2.403732,0.105624,2.196709,2.610754,4


In [4]:
results = summarize_df(df)
results

,estimator,bias,sd,rmse,coverage,avg_ci_length,alpha_y,alpha_d
0,OLS,-0.022152,0.101426,0.102821,0.88,0.353502,0,0
1,DML,-0.016580,0.127722,0.127521,0.88,0.416837,0,0
2,OLS,0.003841,0.095456,0.094575,0.96,0.368994,0,1
3,DML,-0.152367,0.108353,0.186336,0.76,0.431856,0,1
4,OLS,-0.021316,0.176434,0.175957,0.86,0.581567,1,0
5,DML,-0.025111,0.123861,0.125161,0.98,0.461592,1,0
6,OLS,-0.106871,0.129684,0.167042,0.94,0.597999,1,1
7,DML,-0.076784,0.113583,0.136157,0.92,0.476849,1,1


## Plotting Frontier

In [5]:
rmse = rmse_diff(results)

In [7]:
rmse

estimator,alpha_y,alpha_d,DML,OLS,rmse_diff
0,0,0,0.127521,0.102821,-0.024700
1,0,1,0.186336,0.094575,-0.091761
2,1,0,0.125161,0.175957,0.050796
3,1,1,0.136157,0.167042,0.030884


In [8]:
fig = px.density_heatmap(
    rmse,
    x="alpha_d",
    y="alpha_y",
    z="rmse_diff",
    histfunc="avg",
    color_continuous_scale="RdBu",
    title="RMSE Difference: OLS - DML"
)

fig.update_layout(
    xaxis_title="alpha_d",
    yaxis_title="alpha_y",
    coloraxis_colorbar_title="RMSE diff"
)

fig.update_layout(
    xaxis=dict(
        tickmode='array',
        tickvals=rmse.columns,
        ticktext=[str(x) for x in rmse.columns]
    ),
    yaxis=dict(
        tickmode='array',
        tickvals=rmse.index,
        ticktext=[str(y) for y in rmse.index]
    )
)

fig.show()

In [9]:
bias_wide = results.pivot_table(
    index=["alpha_y", "alpha_d"],
    columns="estimator",
    values="bias"
).reset_index()

In [12]:
bias_wide

estimator,alpha_y,alpha_d,DML,OLS
0,0,0,-0.016580,-0.022152
1,0,1,-0.152367,0.003841
2,1,0,-0.025111,-0.021316
3,1,1,-0.076784,-0.106871


In [16]:
fig = px.density_heatmap(bias_wide,
                         y= "alpha_y",
                         x = "alpha_d", 
                         z = "OLS")


fig.update_layout(
    xaxis_title="alpha_d",
    yaxis_title="alpha_y",
    coloraxis_colorbar_title="Bias OLS"
)

fig.update_layout(
    xaxis=dict(
        tickmode='array',
        tickvals=bias_wide.columns,
        ticktext=[str(x) for x in bias_wide.columns]
    ),
    yaxis=dict(
        tickmode='array',
        tickvals=bias_wide.index,
        ticktext=[str(y) for y in bias_wide.index]
    )
)

fig.show()